## <a name="bdd100k">BDD100K数据集</a>

在前面的学习中，我们已经使用COCO数据集训练了实例分割模型。COCO是一个相对比较通用的数据集，为了更好的应对街景交通场景，我们使用更有针对性的BDD100K数据集。BDD100K是伯克利大学发布的大规模开放驾驶视频数据集，数据内容非常丰富，包括针对交通场景下的物体标注，驾驶视频，驾驶线路，GPS信息等。在本案例中，我们主要使用其中的街景物体标注数据集。

### BDD100K标注信息的格式

为了完成目标检测任务，我们使用BDD100K数据集中的images和label两种数据，前者为街景实拍照片，而后者为对照片的标注信息。将BDD100K数据集解压后，得到如下的格式：

```bash
├── bdd100k
│   ├── images
│   │   └── 500
│   │       ├── train
│   │       │   ├── 0195c13b-01c07d6a.jpg
│   │       │   ├── b12ac3f8-be8a7797.jpg
│   │       │   └── xxxxx.jpg
│   │       └── val
│   │           ├── b1d0091f-75824d0d.jpg
│   │           ├── b1d7b3ac-9e14f05f.jpg
│   │           └── xxxxx.jpg
│   └── labels
│       ├── bdd100k_labels_images_train.json
│       └── bdd100k_labels_images_val.json
```

我们取一条BDD100K的标注信息，来查看其json文件的格式：

In [ ]:
import json
import numpy as np

with open('./data/bdd100k/labels/bdd100k_labels_images_val.json', 'r') as f:
    bdd_json = json.load(f)

random_index = np.random.choice(len(bdd_json), 1)[0]
annotation = bdd_json[random_index]

annotation['labels'] = annotation['labels'][:1] # 为了显示方便，我们只取一条标注信息
print(json.dumps(annotation, indent=4))

BDD100K数据标注信息有name, attributes, labels等几个比较重要的字段，分别存储了以下的信息：


|字段|说明|
|---------------------|-------------------------------|
|                name | 图片文件名                      |
|          attributes | 天气，场景，昼夜等信息            |
|              labels | 图片的标注信息                   |
|labels[#]['category] | 物体的分类                      |
|   labels[#]['box2d] | 物体的包围框，其中$(x1, y1)$为左上角坐标，$(x2, y2)为右下角坐标$|

### BDD100K标注数据示例

在本案例中，我们主要使用标注信息中的name，category和labels字段。下面，我们从数据集中随机选取一个标注信息，并将图片中前三个物体的分类和位置标注出来：

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.patches as patches
import matplotlib.pyplot as plt
%matplotlib inline

random_index = np.random.choice(len(bdd_json), 1)[0]
annotation = bdd_json[random_index]
image = Image.open('./data/bdd100k/images/500/val/'+annotation['name'])

num_labels = len(annotation['labels'])
num_to_show = 3 if num_labels > 3 else num_labels

# display the image
plt.figure(figsize=[20, 10])
plt.imshow(image)

# mark the objects in the image with bbox
for i in range(num_to_show):
    # Get the bounding box info
    box = (annotation['labels'][i]['box2d'])
    x1 = box['x1']
    y1 = box['y1']
    width = box['x2'] - x1
    height = box['y2'] - y1
    category = annotation['labels'][i]['category']

    rect = patches.Rectangle((x1, y1), width, height, linewidth=2, edgecolor='b', facecolor='none')
    plt.gca().add_patch(rect)
    plt.gca().text(x1, y1-10, category, fontsize=20, color='b')   

plt.axis('off')
plt.title('Hey, mind the {} objects in the picture'.format(num_to_show), color='r')
plt.show()

这样，我们就对BDD100K数据集中物体检测标注信息的格式有了一定的了解。在后续的学习中，我们对BDD100K数据集进行预处理，对标注格式进行转换，以满足不同模型代码中对格式的要求。